<span style="color: blue">
<h1>Imported Packages</h1>
</span>


In [1]:
import random
from collections import defaultdict
import math

<span style="color: blue">
<h1>Game Class and play_game Function</h1>
</span>


In [2]:
# ============================================================
# Game: Abstract Base Class
# ============================================================
# This class defines the *interface* that any two-player, turn-taking game
# must implement to work with the search algorithms (minimax, alpha-beta,
# etc.). It doesn't play any specific game itself
# Why do it this way? It lets the same minimax/alpha-beta code work for
# ANY game (TicTacToe, Connect Four, etc.) without being rewritten
# the search algorithms only ever call these four methods, never anything
# game-specific.

class Game(object):
    """
    Abstract interface for a two-player game.

    """

    def actions(self, state):
        """
        Return the collection of legal moves available from `state`.

        For Connect Four, this will be "which columns aren't full yet" --
        a move is just a column index (0-6), since a piece always drops
        to the lowest open row in that column.
        """
        raise NotImplementedError  # forces subclasses to implement this -- calling it
                                    # on the base Game class directly is a bug, not
                                    # a valid "no moves" case

    def result(self, state, move):
        """
        Return the NEW state that results from playing `move` in `state`.

        """
        raise NotImplementedError

    def is_terminal(self, state):
        """
        Return True if the game has ended at `state` (win, loss, or draw).

        Default behavior: if there are no legal actions left, the game
        is over. This is a reasonable default for many games (including
        a full Connect Four board), but subclasses often override this
        to ALSO check for a win before the board is full.
        """
        return not self.actions(state)

    def utility(self, state, player):
        """
        Return the value of a TERMINAL state, from `player`'s point of view.

        Convention I will use: +1 = player wins, -1 = player loses, 0 = draw.
        calling it mid-game is undefined.
        """
        raise NotImplementedError

# ============================================================
# play_game: Generic Game Loop
# ============================================================
# This function runs a full game from start to finish, alternating turns
# between whichever "strategies" (move-choosing functions) you give it.
# It doesn't know or care whether the strategies are minimax, alpha-beta,
# a random mover, or a human typing input it just calls whatever
# function is registered for the current player and asks it for a move.

def play_game(game: Game, strategies: dict, verbose=False):
    """
    Play one full game and return the final state.

    Parameters
    ----------
    game : Game
        An instance of a Game subclass (e.g. ConnectFour()). Provides
        the rules via actions/result/is_terminal/utility.
    strategies : dict
        Maps each player identifier -> a function(game, state) -> move.
        Example: {0: player(alphabeta_search), 1: player(random_player)}
        This is how you pit different bot difficulties (or a human)
        against each other without changing this function at all.
    verbose : bool
        If True, print each move and the resulting board as the game
        progresses -- useful for debugging and for visually sanity-checking
        that a difficulty tier is playing the way you expect.

    Returns
    -------
    The final game state, once game.is_terminal(state) is True. Call
    game.utility(final_state, player) afterward to see who won.
    """
    state = game.initial  # start from the game's defined starting position
                           # (e.g. an empty Connect Four board, player 0 to move)

    while not game.is_terminal(state):        # keep playing until someone wins,
                                                # loses, or the board fills up
        player = state.to_move                 # ask the current state whose turn it is
        move = strategies[player](game, state) # hand control to that player's strategy
                                                 # function -- this is where minimax /
                                                 # alpha-beta / random / human input
                                                 # actually gets invoked
        state = game.result(state, move)        # apply the chosen move and advance
                                                 # to the resulting new state

        if verbose:
            print('Player', player, 'move:', move)
            print(state)                        # relies on the state having a
                                                 # __repr__/__str__  we'll define
                                                 # that when we build the board
                                                 # representation

    return state  # the caller can inspect this to determine the winner,
                   # or feed it into game.utility() for a numeric result

<span style="color: blue">
<h1>Minimax Search</h1>
</span>


In [ ]:
# ============================================================
# minimax_search: Exhaustive Game Tree Search (No Pruning)
# ============================================================
# This is the "brute force" baseline: it explores EVERY possible sequence
# of moves all the way to a terminal state (win/loss/draw) before deciding
# on a move. It's correct but slow -- useful as a correctness reference
# and for measuring how much alpha-beta pruning actually saves you.

infinity = math.inf  # shorthand for "worse than any real score" / "better than any real score"

def minimax_search(game, state):
    """
    Search the full game tree from `state` and return (value, move) --
    the best achievable outcome for the player to move, and the move
    that achieves it.

    This uses two mutually recursive helper functions:
      - max_value: called when it's the "maximizing" player's turn
                   (the player we're finding the best move FOR)
      - min_value: called when it's the opponent's turn (assumed to
                   play optimally AGAINST us, i.e. minimize our score)

    This back-and-forth recursion is the core idea of minimax: at every
    level, assume your opponent will make the move that's worst for you,
    and pick the move that's best for you GIVEN that assumption.
    """
    # These globals just count how many times each helper is called,
    # purely for instrumentation/benchmarking -- they have zero effect
    # on the actual search result. Useful for comparing algorithms
    global max_value_calls
    global min_value_calls
    max_value_calls = 0
    min_value_calls = 0

    player = state.to_move  # we are finding the best move FOR this player;
                             # utility() is always evaluated from their
                             # perspective, even inside min_value

    def max_value(state):
        """We are maximizing: try to find the move with the HIGHEST value."""
        global max_value_calls
        max_value_calls += 1

        if game.is_terminal(state):
            # Base case: no more moves to consider, just return how good
            # this ending is for `player`. `move=None` since the game
            # is already over -- there's no move to make.
            return game.utility(state, player), None

        v, move = -infinity, None  # start pessimistic: assume the worst
                                     # possible outcome, so ANY real move
                                     # we find will be an improvement
        for a in game.actions(state):
            # Simulate playing move `a`, then ask: what will the opponent
            # do in response? (min_value handles their turn.) We only
            # care about the resulting VALUE (v2), not what move they'd
            # make -- hence `_` for the discarded move.
            v2, _ = min_value(game.result(state, a))
            if v2 > v:              # found a move that's better for us
                v, move = v2, a      # than anything we've seen so far
        return v, move

    def min_value(state):
        """We are minimizing: the opponent tries to find the LOWEST value
        for `player` (i.e. the best outcome for themselves)."""
        global min_value_calls
        min_value_calls += 1

        if game.is_terminal(state):
            return game.utility(state, player), None

        v, move = +infinity, None  # start optimistic (from `player`'s view):
                                     # assume the opponent can't do any better
                                     # than "infinitely good for player", so
                                     # any real move will be worse (for us)
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a))  # back to our turn
            if v2 < v:               # opponent found a move that hurts
                v, move = v2, a      # `player` more than anything seen so far
        return v, move

    result = max_value(state)  # kick off the recursion from the top,
                                 # always starting from the maximizing
                                 # player's perspective (the one whose
                                 # turn it currently is)

    print(f'minimax_search end - max_value_calls is {max_value_calls} and min_value_calls is {min_value_calls}')
    print(f'minimax_search end - utility is {result[0]} and move is {result[1]}\n')

    return result


# ============================================================
# alphabeta_search: Minimax + Alpha-Beta Pruning
# ============================================================
# Same recursive max/min structure as minimax_search, but now we carry
# two extra values, alpha and beta, that let us SKIP exploring branches
# that can't possibly change the final decision. This produces the exact
# same answer as minimax_search, just faster -- pruning never changes
# correctness, only how much of the tree gets visited.
#
#   alpha = the best value the MAXIMIZING player can guarantee so far
#           (a lower bound on the final result)
#   beta  = the best value the MINIMIZING player can guarantee so far
#           (an upper bound on the final result)
#
# The key insight: once we find a branch where v >= beta, we know the
# minimizing player (earlier in the tree) would never let us reach this
# branch anyway -- they already have a better (lower) option elsewhere.
# So there's no point searching this branch's remaining siblings; we can
# stop early ("prune"). This is called a "beta cutoff". The symmetric
# case (v <= alpha) happens inside min_value.

def alphabeta_search(game, state):
    """
    Same contract as minimax_search: returns (value, move). Produces
    IDENTICAL results to minimax_search, just typically visits far
    fewer nodes
    """
    global max_value_calls
    global min_value_calls
    max_value_calls = 0
    min_value_calls = 0

    player = state.to_move

    def max_value(state, alpha, beta):
        global max_value_calls
        max_value_calls += 1

        if game.is_terminal(state):
            return game.utility(state, player), None

        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta)
            if v2 > v:
                v, move = v2, a
                alpha = max(v, alpha)  
            if v >= beta:
                # Cutoff: the minimizing player up the tree already has a
                # way to hold us to `beta` or worse. Since we've already
                # found something >= beta here, they'll never route the
                # game into this branch -- so checking the REMAINING
                # sibling moves is provably wasted work. Stop and return.
                return v, move
        return v, move

    def min_value(state, alpha, beta):
        global min_value_calls
        min_value_calls += 1

        if game.is_terminal(state):
            return game.utility(state, player), None

        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta)
            if v2 < v:
                v, move = v2, a
                beta = min(v, beta)  # tighten our upper bound: the opponent
                                       # can now guarantee holding us to AT
                                       # MOST this value
            if v <= alpha:
                # Symmetric cutoff: the maximizing player up the tree
                # already has a way to guarantee at least `alpha`. Since
                # this branch has dropped to alpha or below, they'll never
                # let the game come this way -- stop exploring siblings.
                return v, move
        return v, move

    result = max_value(state, -infinity, +infinity)  # search starts with
                                                        # the widest possible
                                                        # window: nothing is
                                                        # pruned yet

    print(f'alphabeta_search end - max_value_calls is {max_value_calls} and min_value_calls is {min_value_calls}')
    print(f'alphabeta_search end - utility is {result[0]} and move is {result[1]}\n')

    return result


# ============================================================
# cutoff_depth + h_alphabeta_search: Depth-Limited Alpha-Beta
# with a Heuristic Evaluation
# ============================================================
# Full alpha-beta is still exhaustive -- it searches all the way to
# terminal states, just skipping provably-irrelevant branches. For a
# game like Connect Four, "all the way to terminal" can be a HUGE tree.
# h_alphabeta_search adds a depth limit: once we've looked `d` moves
# ahead without the game ending, we stop and estimate how good the
# position looks using a heuristic function `h`, instead of continuing
# to search.
#
# IMPORTANT GOTCHA (learned from our own benchmarking): the default
# heuristic below, `h=lambda s, p: 0`, always returns 0 for every
# non-terminal position. This means the search has NO way to judge
# "this position looks promising" vs "this position looks bad" until
# it actually reaches a real win/loss -- which cripples alpha-beta's
# pruning power, since pruning depends on being able to tell branches
# apart. Always pass a REAL heuristic function when calling this for
# anything beyond a toy example, or you'll get much worse performance
# than alpha-beta should be capable of.

def cutoff_depth(d):
    """
    Returns a cutoff-test FUNCTION (not a value) that says "stop searching
    once we're more than `d` plies deep." This is a small bit of Python
    trickery called a "closure" -- cutoff_depth(6) builds and returns a
    new function that remembers the value 6 forever, without needing a
    global variable.

    Usage: cutoff=cutoff_depth(8) means "search 8 moves ahead, then stop
    and evaluate with the heuristic."
    """
    return lambda game, state, depth: depth > d


def h_alphabeta_search(game, state, cutoff=cutoff_depth(6), h=lambda s, p: 0):
    """
    Depth-limited alpha-beta search with a pluggable heuristic.

    Parameters
    ----------
    cutoff : function(game, state, depth) -> bool
        Decides when to stop searching deeper and fall back to the
        heuristic instead. Build one with cutoff_depth(d).
    h : function(state, player) -> number
        Estimates how good `state` looks for `player`, WITHOUT actually
        finishing the game out. This is where real game knowledge goes
        (e.g. for Connect Four: center-column control, counting
        potential 4-in-a-rows, etc.) -- the quality of this function
        has a huge effect on both playing strength AND search speed
        (a heuristic that can tell moves apart enables much better
        pruning, not just better move choices).

    Returns (value, move), same contract as the other two search
    functions above.
    """
    global max_value_calls
    global min_value_calls
    max_value_calls = 0
    min_value_calls = 0

    player = state.to_move

    def max_value(state, alpha, beta, depth):
        global max_value_calls
        max_value_calls += 1

        if game.is_terminal(state):
            # Real game-over state: we know the TRUE outcome, no need
            # to guess with the heuristic.
            return game.utility(state, player), None
        if cutoff(game, state, depth):
            # We've hit our search-depth budget without the game ending.
            # Stop recursing and estimate the value instead -- this is
            # what makes deep games like Connect Four tractable at all.
            return h(state, player), None

        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta, depth + 1)  # note: depth+1,
                                                                                 # tracking how far
                                                                                 # we've descended
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

    def min_value(state, alpha, beta, depth):
        global min_value_calls
        min_value_calls += 1

        if game.is_terminal(state):
            return game.utility(state, player), None
        if cutoff(game, state, depth):
            return h(state, player), None

        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta, depth + 1)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    result = max_value(state, -infinity, +infinity, 0)  # depth starts at 0
                                                            # at the root

    print(f'h_alphabeta_search end - max_value_calls is {max_value_calls} and min_value_calls is {min_value_calls}')
    print(f'h_alphabeta_search end - utility is {result[0]} and move is {result[1]}\n')

    return result


# ============================================================
# Board: Generic Game Board Storage
# ============================================================
# Represents a board as a dictionary mapping (x, y) coordinates to
# whatever's on that square ('X', 'O', or nothing). Subclassing
# `defaultdict` means we get free handling of "what's at a square I've
# never touched" via __missing__ below, instead of KeyErrors everywhere.
#
# NOTE: this dict-based representation is fine for a Python test suite
# where we're validating algorithm CORRECTNESS, but it's not what we
# want for the embedded C port -- copying a dict on every simulated move
# (see `new()` below) is exactly the overhead a bitboard representation
# is designed to eliminate.

class Board(defaultdict):
    """
    A board has:
      - a width and height (board dimensions)
      - `to_move`: whose turn it is next
      - `utility`: cached result of the game (0 = ongoing/draw, +1/-1 = win/loss)
      - dict entries of the form (x, y): 'X' or (x, y): 'O' for occupied squares
    Unoccupied squares simply aren't in the dict at all -- see __missing__.
    """
    empty = '.'  # symbol shown for an empty, in-bounds square
    off = '#'    # symbol shown for a square outside the board entirely
                 # (useful for win-checking code that scans past the edges)

    def __init__(self, width=8, height=8, to_move=None, **kwds):
        # `self.__dict__.update(...)` is a compact way of doing
        # self.width = width; self.height = height; self.to_move = to_move;
        # plus anything extra passed in via **kwds (like utility=0).
        # We use this instead of individual assignments purely for brevity.
        self.__dict__.update(width=width, height=height, to_move=to_move, **kwds)

    def new(self, changes: dict, **kwds) -> 'Board':
        """
        Return a BRAND NEW Board reflecting `changes` applied on top of
        this one, WITHOUT modifying this board.

        This immutability is essential for the search algorithms above:
        minimax/alpha-beta explore many hypothetical futures via
        game.result(state, move), and if result() mutated the existing
        board in place, earlier recursive calls that still hold a
        reference to "the old state" would see it corrupted out from
        under them.
        """

        """
          **kwds: a "grab bag" for extra named info we want the new Board
          to have (like to_move= or utility=) that isn't a piece placement.
          Anything passed in here gets scooped up into the kwds dict, then
          dumped back out below via Board(..., **kwds) to set those same
          values on the fresh board.
        """
        board = Board(width=self.width, height=self.height, **kwds)  # fresh board,
                                                                        # same dimensions,
                                                                        # new to_move/utility
                                                                        # via kwds
        board.update(self)      # copy every existing (x,y):piece entry from
                                  # the current board into the new one
        board.update(changes)   # then overlay the new move(s) on top --
                                  # this is the O(n) copy that dominates cost
                                  # in the Python version (n = pieces placed
                                  # so far); a bitboard just ORs in one bit
                                  # instead
        return board

    def __missing__(self, loc):
        """
        Called automatically by defaultdict whenever you look up a key
        (x, y) that isn't actually stored in the dict -- i.e. an empty
        square. Rather than raising a KeyError, we compute what SHOULD
        be shown there.
        """
        x, y = loc
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.empty  # in bounds, just not played yet
        else:
            return self.off    # out of bounds entirely (used by
                                 # win-detection code that probes
                                 # neighboring squares without first
                                 # checking board edges)

    def __hash__(self):
        """
        Makes Board objects usable as dict keys / in sets (e.g. for a
        transposition table later on). Two boards with the same pieces
        in the same places and the same player to move should hash the
        same way, regardless of the ORDER pieces were inserted in the
        underlying dict -- hence sorting the items before hashing.
        """
        return hash(tuple(sorted(self.items()))) + hash(self.to_move)

    def __repr__(self):
        """
        Human-readable board printout, used whenever you `print(board)`
        (including inside play_game's verbose mode).
        """
        def row(y):
            return ' '.join(self[x, y] for x in range(self.width))
        return '\n'.join(map(row, range(self.height))) + '\n'


# ============================================================
# k_in_row: Win Detection Helper
# ============================================================
# Checks whether placing a piece at `square` completed a line of `k`
# same-player pieces, in any of the four possible directions (horizontal,
# vertical, and both diagonals). This only needs to check lines running
# THROUGH the just-played square -- it doesn't need to re-scan the whole
# board, since the move just played is the only way a NEW winning line
# could have just formed.

def k_in_row(board, player, square, k):
    """
    Return True if `player` has `k` pieces in an unbroken line running
    through `square`, in any direction.
    """
    def in_row(x, y, dx, dy):
        # Walk outward from (x, y) in direction (dx, dy) one step at a
        # time, counting how many consecutive squares belong to `player`.
        # board[x, y] relies on Board.__missing__ above to safely return
        # '.' or '#' for empty/out-of-bounds squares without crashing.
        if board[x, y] != player:
            return 0
        return 1 + in_row(x + dx, y + dy, dx, dy)

    (x, y) = square
    return any(
        # For each of the 4 line orientations, count consecutive pieces
        # extending BOTH ways from the played square, then add 1 for the
        # square itself. E.g. for horizontal: count how far the line
        # extends to the right (dx=1) plus how far it extends to the
        # left (dx=-1), plus the piece just placed.
        in_row(x, y, 0, 1) + in_row(x, y, 0, -1) - 1 >= k or   # vertical
        in_row(x, y, 1, 0) + in_row(x, y, -1, 0) - 1 >= k or   # horizontal
        in_row(x, y, 1, 1) + in_row(x, y, -1, -1) - 1 >= k or  # diagonal "/"
        in_row(x, y, 1, -1) + in_row(x, y, -1, 1) - 1 >= k     # diagonal "\"
        for (x, y) in [square]
    )


# ============================================================
# TicTacToe: A Concrete Game Implementation
# ============================================================
# This fills in the Game interface for TicTacToe (and, more generally,
# for "k-in-a-row on an arbitrary height x width grid" -- which is
# exactly why ConnectFour can inherit from it below instead of
# duplicating this logic).

class TicTacToe(Game):
    """
    Play k-in-a-row on a `height` x `width` board. 'X' always moves first,
    against 'O'. With the defaults (3x3, k=3), this is standard TicTacToe.
    """

    def __init__(self, height=3, width=3, k=3):
        self.k = k  # how many in a row needed to win (3 for TicTacToe,
                     # 4 for Connect Four)
        self.squares = {(x, y) for x in range(width) for y in range(height)}
        # The complete set of every valid coordinate on the board --
        # used below to figure out which squares are still empty.
        self.initial = Board(height=height, width=width, to_move='X', utility=0)
        # The starting position: an empty board, X to move, utility 0
        # (meaning "no result yet").

    def actions(self, board):
        """
        Legal moves = any square not yet occupied.
        `set(board)` gives the keys of the board dict (i.e. squares
        that HAVE been played), so squares - set(board) is exactly
        "all squares minus played squares" = empty squares.
        """
        return list(self.squares - set(board))

    def result(self, board, square):
        """
        Return the new board state after placing the current player's
        piece at `square`.
        """
        player = board.to_move
        # Build the new board: current player's mark goes at `square`,
        # and it becomes the OTHER player's turn.
        board = board.new({square: player}, to_move=('O' if player == 'X' else 'X'))
        win = k_in_row(board, player, square, self.k)
        # Cache the outcome on the new board so is_terminal/utility
        # don't need to re-derive it later: 0 = no win yet, +1 = X won,
        # -1 = O won. (This convention -- always relative to X -- is
        # why utility() below flips the sign for player == 'O'.)
        board.utility = (0 if not win else +1 if player == 'X' else -1)
        return board

    def utility(self, board, player):
        """
        Convert the board's X-relative cached utility into `player`'s
        own perspective: if you're O, a win for X (utility=+1) is
        actually a LOSS for you, hence the sign flip.
        """
        return board.utility if player == 'X' else -board.utility

    def is_terminal(self, board):
        """
        Game over if someone already won (utility != 0), OR if every
        square is filled (a draw). len(board) counts dict entries, i.e.
        squares that have been played.
        """
        return board.utility != 0 or len(self.squares) == len(board)

    def display(self, board):
        print(board)


# ============================================================
# random_player / player: Strategy Functions for play_game
# ============================================================
# play_game expects strategies as {player_name: function(game, state) -> move}.
# These two helpers build such functions.

def random_player(game, state):
    """
    A strategy that picks a uniformly random legal move -- useful as a
    weak baseline opponent (e.g. for sanity-testing that a bot reliably
    beats random play) and as a fast stand-in when you don't want the
    overhead of a real search on one side of a benchmark.

    Returns (0, move) to match the (value, move) shape that play_game's
    strategy functions are expected to produce -- the 0 is a throwaway
    value never actually used, since play_game only reads the move.
    """
    return 0, random.choice(list(game.actions(state)))


def player(search_algorithm):
    """
    Wraps any of our search functions (minimax_search, alphabeta_search,
    h_alphabeta_search) into a strategy function play_game can call.

    Why the wrapping? play_game calls strategies[player](game, state) and
    expects just a MOVE back, but our search functions return (value, move)
    tuples (since the value is useful for debugging/benchmarking). This
    lambda calls the search function and throws away the value, keeping
    only [1], the move -- adapting the search function's richer return
    type to the simpler interface play_game expects.

    Usage: dict(X=player(alphabeta_search), O=player(h_alphabeta_search))
    """
    return lambda game, state: search_algorithm(game, state)[1]


# ============================================================
# ConnectFour: Reusing TicTacToe's Rules for a Bigger Board
# ============================================================
# Connect Four is "7 wide x 6 tall, 4-in-a-row to win" -- which is
# exactly what TicTacToe already implements generically! The ONLY real
# rule difference is gravity: in TicTacToe you can place a piece on any
# empty square, but in Connect Four a piece always falls to the lowest
# empty row in its chosen column. So we only need to override `actions`
# (what counts as a legal move) -- everything else (result, utility,
# is_terminal, win-checking via k_in_row) is inherited unchanged.

class ConnectFour(TicTacToe):
    """
    Standard Connect Four: 7 columns wide, 6 rows tall, 4 in a row to win.
    """

    def __init__(self):
        # k=4 (four in a row wins), 7 wide, 6 tall -- reuses ALL of
        # TicTacToe's __init__ logic (building self.squares, self.initial)
        # just with different dimensions/win-length.
        super().__init__(height=6, width=7, k=4)

    def actions(self, board):
        """
        Legal moves in Connect Four = one potential move PER COLUMN
        (not per empty square) -- the lowest empty row in each
        non-full column, since gravity means you can't choose where
        in the column your piece lands.
        """
        potent_moves = set()
        for x in range(board.width):                       # check each column...
            for y in reversed(range(board.height)):         # ...starting from the
                                                               # BOTTOM row (largest y)
                                                               # and moving up, since
                                                               # that's the order pieces
                                                               # actually stack
                if board[x, y] == board.empty:
                    potent_moves.add((x, y))  # first empty square found going
                                                # bottom-up is where a piece
                                                # dropped in this column would land
                    break                       # stop scanning this column --
                                                  # only ONE legal move per column
        return potent_moves

<span style="color: blue">
<h1>Algorithm Optimization (Heuristic, depth discount, etc.)</h1>
</span>


In [ ]:
# ============================================================
# connect4_heuristic: Positional Evaluation for Non-Terminal Cutoffs
# ============================================================
# Used whenever h_alphabeta_search hits its depth limit before the game
# actually ends -- instead of returning a meaningless flat 0, this scores
# how promising the position LOOKS, based on two ideas:
#   1. Center-column control: pieces in the middle column participate in
#      more possible 4-in-a-row lines than edge pieces, so they're worth
#      more even before any line is close to complete.
#   2. Windows: every possible 4-in-a-row line on the board ("window")
#      is scored based on how many of each player's pieces are already
#      in it -- more of YOUR pieces with no opponent pieces blocking it
#      is good, more of the OPPONENT'S pieces is bad (weighted more
#      heavily, since blocking threats matters more than building your
#      own in most Connect Four positions).

def window_score(pieces, player, opp):
    """
    Score a single 4-cell window based on what's already placed in it.
    `pieces` is a list of 4 cell contents (e.g. ['X', '.', 'X', '.']).
    """
    p_count = pieces.count(player)
    o_count = pieces.count(opp)

    if p_count > 0 and o_count > 0:
        return 0  # window has BOTH players' pieces in it -- nobody can
                   # ever complete 4-in-a-row here, so it's worthless to
                   # either side; skip it entirely

    if p_count == 4:
        return 1_000_000   # already-won window (only reached if this
                             # heuristic is ever called on a state we
                             # somehow didn't already catch as terminal --
                             # a large value guards against that edge case)
    if p_count == 3:
        return 50           # three in a row with the 4th cell open --
                              # a real, immediate threat, weighted heavily
    if p_count == 2:
        return 5             # a building block -- worth something, but
                               # far less urgent than a 3-in-a-row threat
    if o_count == 3:
        return -80            # OPPONENT has 3 in a row -- weighted MORE
                                # negatively than our own 3-in-a-row is
                                # weighted positively. This encodes "block
                                # first, build second": at equal search
                                # depth, we want the bot to prioritize
                                # stopping the opponent's near-win over
                                # advancing its own slightly-behind threat
    if o_count == 2:
        return -4
    return 0  # empty window, or just 1 piece in it -- not meaningfully
               # informative yet


def connect4_heuristic(board, player):
    """
    Estimate how good `board` looks for `player`, without playing the
    game out. Called only when h_alphabeta_search's depth cutoff is
    reached on a NON-terminal position -- terminal wins/losses/draws
    always use the real utility() instead, since that's exact and this
    is only an estimate.
    """
    opp = 'O' if player == 'X' else 'X'
    score = 0
    W, H = board.width, board.height

    # --- Center column control ---
    # Column 3 (0-indexed, middle of 7) participates in more possible
    # winning lines (horizontal, both diagonals) than any edge column,
    # so pieces placed there are worth a flat bonus regardless of
    # whether any specific line is close to completion yet.
    center_col = W // 2
    center_count = sum(1 for y in range(H) if board[center_col, y] == player)
    score += center_count * 6

    # --- Build the list of every possible 4-in-a-row "window" on the board ---
    # This is recomputed every call for clarity here; for a real embedded
    # port, this list of window COORDINATES is fixed for a given board
    # size and should be precomputed ONCE (or replaced with bitmask
    # constants), not rebuilt every evaluation.
    lines = []
    for y in range(H):                      # horizontal windows
        for x in range(W - 3):
            lines.append([(x + i, y) for i in range(4)])
    for x in range(W):                      # vertical windows
        for y in range(H - 3):
            lines.append([(x, y + i) for i in range(4)])
    for x in range(W - 3):                  # diagonal windows (both directions)
        for y in range(H - 3):
            lines.append([(x + i, y + i) for i in range(4)])       # "/" diagonal
            lines.append([(x + i, y + 3 - i) for i in range(4)])   # "\" diagonal

    for line in lines:
        pieces = [board[pt] for pt in line]
        score += window_score(pieces, player, opp)

    return score


# ============================================================
# ConnectFour: Center-First Move Ordering
# ============================================================
# Replaces the set-based actions() with a deterministic, center-out
# ORDERED list. This does two things: (1) makes search behavior
# reproducible instead of depending on incidental set-iteration order
# improves actual pruning, since trying strong
# moves first lets alpha-beta cut off bad branches sooner.

class ConnectFour(TicTacToe):
    def __init__(self):
        super().__init__(height=6, width=7, k=4)
        # Precompute column search order once: center column (3) first,
        # then alternating outward -- [3, 2, 4, 1, 5, 0, 6]
        self._col_order = sorted(range(7), key=lambda c: abs(c - 3))

    def actions(self, board):
        potent_moves = []  # a LIST now, not a set -- order is
                             # intentional and must be preserved
        for x in self._col_order:
            for y in reversed(range(board.height)):
                if board[x, y] == board.empty:
                    potent_moves.append((x, y))
                    break
        return potent_moves


# ============================================================
# h_alphabeta_search: now with depth-discounted terminal values
# ============================================================
# Same as before, but wins/losses found SOONER are scored strictly
# better/worse than the same outcome found later
DEPTH_DISCOUNT = 0.01  # safe for depths up to ~42 (full game length)

def h_alphabeta_search(game, state, cutoff=cutoff_depth(6), h=connect4_heuristic):
    global max_value_calls, min_value_calls
    max_value_calls = 0
    min_value_calls = 0
    player = state.to_move

    def terminal_value(state, depth):
        raw = game.utility(state, player)
        return raw - (DEPTH_DISCOUNT * depth if raw != 0 else 0)

    def max_value(state, alpha, beta, depth):
        global max_value_calls
        max_value_calls += 1
        if game.is_terminal(state):
            return terminal_value(state, depth), None
        if cutoff(game, state, depth):
            return h(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta, depth + 1)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

    def min_value(state, alpha, beta, depth):
        global min_value_calls
        min_value_calls += 1
        if game.is_terminal(state):
            return terminal_value(state, depth), None
        if cutoff(game, state, depth):
            return h(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta, depth + 1)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    result = max_value(state, -infinity, +infinity, 0)
    print(f'h_alphabeta_search end - max_value_calls is {max_value_calls} and min_value_calls is {min_value_calls}')
    print(f'h_alphabeta_search end - utility is {result[0]} and move is {result[1]}\n')
    return result

<span style="color: blue">
<h1>Benchmark Stats</h1>
</span>


In [ ]:
# ============================================================
# Benchmark Harness: Time + Node-Count a Single Search Call
# ============================================================
# This wraps any of our three search functions (minimax_search,
# alphabeta_search, h_alphabeta_search) with wall-clock timing and pulls
# out the node counts they already track internally (max_value_calls /
# min_value_calls).

import time  # perf_counter() gives a high-resolution wall-clock timer,
             # more precise than time.time() for measuring short-running
             # code -- exactly what we need since some of these searches
             # finish in milliseconds and others take minutes

def benchmark_search(label, search_fn, game, state, **kwargs):
    """
    Run one search call, time it, and report the results in a
    consistent format across algorithms.

    Parameters
    ----------
    label : str
        A human-readable name for this run, e.g. "alphabeta (no limit)"
        or "h_alphabeta depth=8" -- just for the printout.
    search_fn : function
        One of minimax_search, alphabeta_search, or h_alphabeta_search.
    game, state : Game, Board
        The game rules and the specific position to search from.
    **kwargs
        Passed straight through to search_fn -- this is how you supply
        `cutoff=cutoff_depth(8)` or `h=my_heuristic` for h_alphabeta_search
        without this function needing to know about those parameters
        specifically.

    Returns
    -------
    A dict with the value/move found, elapsed time, node counts, and
    a derived nodes/sec throughput -- handy for collecting into a list
    and comparing across configurations (see the depth-sweep cell below).
    """
    start = time.perf_counter()                     # mark start time
    value, move = search_fn(game, state, **kwargs)   # run the actual search --
                                                        # this also sets the global
                                                        # max_value_calls/min_value_calls
                                                        # counters as a side effect
    elapsed = time.perf_counter() - start            # wall-clock seconds elapsed

    total_nodes = max_value_calls + min_value_calls  # combined node count,
                                                        # reading the globals that
                                                        # search_fn just populated
    nodes_per_sec = total_nodes / elapsed if elapsed > 0 else float('inf')
    # Guard against division by zero for searches so fast they round to 0.0s

    print(f"--- {label} ---")
    print(f"  value={value}, move={move}")
    print(f"  max_value_calls={max_value_calls:,}  min_value_calls={min_value_calls:,}  total_nodes={total_nodes:,}")
    print(f"  elapsed={elapsed:.4f}s   throughput={nodes_per_sec:,.0f} nodes/sec\n")

    return {
        "label": label, "value": value, "move": move,
        "elapsed_sec": elapsed, "total_nodes": total_nodes,
        "nodes_per_sec": nodes_per_sec,
    }


# ============================================================
# Depth Sweep: See How Search Cost Grows with Depth
# ============================================================
# This is the single most useful cell for our ESP32 planning: it runs
# h_alphabeta_search at several depths on the SAME starting position and
# shows me exactly how nodes/time blow up as depth increases. This is
# how we empirically find "the deepest search that still fits in our
# time budget" instead of guessing.
#
# NOTE: we deliberately do NOT include plain minimax_search or unlimited
# alphabeta_search here for ConnectFour -- from an empty board, both
# would search toward the full 7^42-scale game tree with no depth cutoff,
# which is not something you want to wait for.

game = ConnectFour()
state = game.initial   # empty board -- the single hardest position to
                        # search, since the most future moves remain.
                        # This matches the "worst case" framing we used
                        # earlier when projecting ESP32 timing.

depths_to_test = [2, 4, 6, 8, 9]   # add higher values cautiously --
                                     # cost grows roughly exponentially,
                                     # so depth 10-12 may take a while
                                     # without a real heuristic (see note
                                     # below)

results = []
for d in depths_to_test:
    result = benchmark_search(
        label=f"h_alphabeta depth={d}",
        search_fn=h_alphabeta_search,
        game=game,
        state=state,
        cutoff=cutoff_depth(d),
        # h= intentionally omitted here -- this uses the default h=lambda s,p: 0,
        # Once i write a real heuristic function, rerun this same sweep
        # with h=my_heuristic added below and compare the node counts --
        # that side-by-side comparison is the clearest way to see exactly
        # how much the heuristic saves us.
    )
    results.append(result)

# Quick summary table across all depths tested, so we can see the growth
# trend at a glance instead of scrolling back through printed output.
print(f"{'Depth':<8}{'Nodes':<15}{'Time (s)':<12}{'Nodes/sec':<15}")
for d, r in zip(depths_to_test, results):
    print(f"{d:<8}{r['total_nodes']:<15,}{r['elapsed_sec']:<12.4f}{r['nodes_per_sec']:<15,.0f}")

h_alphabeta_search end - max_value_calls is 14 and min_value_calls is 68
h_alphabeta_search end - utility is 17 and move is (3, 5)

--- h_alphabeta depth=2 ---
  value=17, move=(3, 5)
  max_value_calls=14  min_value_calls=68  total_nodes=82
  elapsed=0.0066s   throughput=12,372 nodes/sec

h_alphabeta_search end - max_value_calls is 219 and min_value_calls is 1008
h_alphabeta_search end - utility is 23 and move is (3, 5)

--- h_alphabeta depth=4 ---
  value=23, move=(3, 5)
  max_value_calls=219  min_value_calls=1,008  total_nodes=1,227
  elapsed=0.1248s   throughput=9,831 nodes/sec

h_alphabeta_search end - max_value_calls is 1487 and min_value_calls is 6634
h_alphabeta_search end - utility is 38 and move is (3, 5)

--- h_alphabeta depth=6 ---
  value=38, move=(3, 5)
  max_value_calls=1,487  min_value_calls=6,634  total_nodes=8,121
  elapsed=0.9833s   throughput=8,259 nodes/sec

h_alphabeta_search end - max_value_calls is 13814 and min_value_calls is 44806
h_alphabeta_search end - utili

<span style="color: blue">
<h1>Human VS AI PlayTest</h1>
</span>


In [ ]:
# ============================================================
# Interactive Play: Human vs. AI, with Selectable Difficulty
# ============================================================
# Self-contained version: includes human_player directly, so this cell
# runs correctly even if executed on its own, without depending on
# earlier cells having been run first

import random  # safe to re-import even if already imported earlier

# ------------------------------------------------------------
# human_player: prompts you for a move each turn
# ------------------------------------------------------------
def human_player(game, state):
    """
    Prompt the human for a move, re-asking until a legal one is given.
    Expects input as "x y" (column, row) -- x is 0-6 (column), y is
    0-5 (row), matching the (x, y) move format the rest of the code uses.
    """
    legal_moves = list(game.actions(state))
    print(f"Available moves: {sorted(legal_moves)}")
    while True:
        try:
            move = tuple(map(int, input("Enter your move as x y (e.g., 3 5): ").strip().split()))
            if move in legal_moves:
                return move
            print("Invalid move. Try again.")
        except Exception:
            print("Invalid input. Try again, format must be: x y")


# ------------------------------------------------------------
# Difficulty presets (Python test-suite version)
# ------------------------------------------------------------
DIFFICULTY_PRESETS = {
    "noob":   {"max_depth": 2,  "blunder_chance": 0.25, "use_opening_book": False},
    "pro":    {"max_depth": 8,  "blunder_chance": 0.0,  "use_opening_book": False},
    "hacker": {"max_depth": 10, "blunder_chance": 0.0,  "use_opening_book": True},
}

def get_bot_move(game, state, cfg):
    """
    Decide the bot's move for this turn, given a difficulty config dict.
    Same three-step logic regardless of difficulty: opening book check,
    then blunder roll, then real search -- difficulty is a config, not
    a different algorithm.
    """
    if cfg["use_opening_book"] and len(state) == 0:
        return (3, 5)  # center column, bottom row -- strongest known
                         # Connect Four opening move

    if random.random() < cfg["blunder_chance"]:
        return random.choice(list(game.actions(state)))

    value, move = h_alphabeta_search(game, state, cutoff=cutoff_depth(cfg["max_depth"]))
    return move


# ------------------------------------------------------------
# Difficulty selection
# ------------------------------------------------------------
print("Choose a difficulty: noob / pro / hacker")
difficulty_choice = input("Difficulty: ").strip().lower()

while difficulty_choice not in DIFFICULTY_PRESETS:
    print("Not a valid option.")
    difficulty_choice = input("Choose noob, pro, or hacker: ").strip().lower()

cfg = DIFFICULTY_PRESETS[difficulty_choice]
print(f"\nStarting a game on '{difficulty_choice}' difficulty "
      f"(depth={cfg['max_depth']}, blunder_chance={cfg['blunder_chance']})\n")

# ------------------------------------------------------------
# Play the game
# ------------------------------------------------------------
game = ConnectFour()
strategies = dict(X=human_player, O=lambda g, s: get_bot_move(g, s, cfg))

start_time = time.perf_counter()  # simple timer instead of the %time magic --
                                    # avoids the multi-line-magic quirk entirely
final_state = play_game(game, strategies, verbose=True).utility
elapsed = time.perf_counter() - start_time

print(f"\nTotal game time: {elapsed:.2f}s")

# ------------------------------------------------------------
# Result
# ------------------------------------------------------------
print("Final utility value:", final_state)
if final_state > 0:
    print("You (X) won!")
elif final_state < 0:
    print(f"The computer (O, {difficulty_choice}) won!")
else:
    print("It's a draw!")

Choose a difficulty: noob / pro / hacker


Difficulty:  hacker



Starting a game on 'hacker' difficulty (depth=10, blunder_chance=0.0)

Available moves: [(0, 5), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (6, 5)]


Enter your move as x y (e.g., 3 5):  0 5


Player X move: (0, 5)
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
X . . . . . .

h_alphabeta_search end - max_value_calls is 162712 and min_value_calls is 512036
h_alphabeta_search end - utility is 24 and move is (3, 5)

Player O move: (3, 5)
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
X . . O . . .

Available moves: [(0, 4), (1, 5), (2, 5), (3, 4), (4, 5), (5, 5), (6, 5)]


Enter your move as x y (e.g., 3 5):  1 5


... (11 outputs left out)

Enter your move as x y (e.g., 3 5):  3 2


Player X move: (3, 2)
. X . . . . .
. O . . . . .
O O . X . . .
X O . O . . .
X X . O . . .
X X . O . . X

h_alphabeta_search end - max_value_calls is 67845 and min_value_calls is 142524
h_alphabeta_search end - utility is 39 and move is (3, 1)

Player O move: (3, 1)
. X . . . . .
. O . O . . .
O O . X . . .
X O . O . . .
X X . O . . .
X X . O . . X

Available moves: [(0, 1), (2, 5), (3, 0), (4, 5), (5, 5), (6, 4)]


Enter your move as x y (e.g., 3 5):  2 5


Player X move: (2, 5)
. X . . . . .
. O . O . . .
O O . X . . .
X O . O . . .
X X . O . . .
X X X O . . X

h_alphabeta_search end - max_value_calls is 10536 and min_value_calls is 20682
h_alphabeta_search end - utility is 0.99 and move is (2, 4)

Player O move: (2, 4)
. X . . . . .
. O . O . . .
O O . X . . .
X O . O . . .
X X O O . . .
X X X O . . X


Total game time: 1917.26s
Final utility value: -1
The computer (O, hacker) won!
